# 05 — LSTM Autoencoder

## 1. Objective

Develop an LSTM autoencoder for unsupervised anomaly detection on the SWaT multivariate time series.

The model will be trained using normal calibration data and will learn to reconstruct temporal patterns in the sensor measurements. Reconstruction error will then provide an anomaly score: observations that are poorly reconstructed relative to the learned normal behaviour may indicate anomalous behaviour.

The implementation follows the LSTM autoencoder approach used in the primary reference notebook, while maintaining the project's requirements for chronological data handling, separation of calibration and evaluation data, reproducibility, and prevention of data leakage.

Attack labels will not be used during model training or reconstruction-error threshold calibration.

### Scope of Evaluation

This notebook evaluates the LSTM autoencoder on the normal validation period using the same eligible-normal split as Notebook 04. Attack-side evaluation — including confusion-matrix counts, precision, recall, and F1 on attack-labelled sequences — is performed in a subsequent stage, following the deferral recorded in Notebook 04 §13.3. The temporal overlap between the attack file and the normal time windows, established by Notebook 02's normal–attack timestamp overlap analysis, must be respected when that evaluation is carried out.

## 2. Methodology

An autoencoder is trained to reconstruct input observations from a compressed internal representation. When trained only on normal process behaviour, the model is expected to learn the temporal patterns present in normal operation.

For an input sequence X, the LSTM encoder maps the sequence to a lower-dimensional representation:

X → Encoder → Z

The decoder then reconstructs the sequence:

Z → Decoder → X̂

The reconstruction error measures the difference between the original and reconstructed sequence. For a sequence containing T time steps and D sensor features, the mean squared reconstruction error is:

MSE = (1 / (T × D)) × Σ (Xₜ,ⱼ − X̂ₜ,ⱼ)²

where:

- Xₜ,ⱼ is the observed value for feature j at time step t.
- X̂ₜ,ⱼ is the corresponding reconstructed value.
- T is the sequence length.
- D is the number of sensor features.

A threshold derived from the reconstruction-error distribution on held-out normal calibration observations is then used to classify sequences as normal or anomalous. Following the percentile approach used by the simple statistical baseline in Notebook 04 Section 12, the threshold is set at the 99th percentile of reconstruction errors on a portion of the calibration period held out from model training. The threshold is fixed before any attack sequence is scored, and attack labels are not used to select or adjust it. The specific hold-out split used for threshold calibration is recorded alongside the model-training code.

The per-sequence reconstruction error is defined as:

$$
e(\mathbf{X}_i) = \frac{1}{T \cdot D} \sum_{t=1}^{T} \sum_{j=1}^{D} \left( X_{i,t,j} - \hat{X}_{i,t,j} \right)^2
$$

where $\mathbf{X}_i$ is the $i$-th sequence, $T$ is the sequence length, $D$ is the number of features, $X_{i,t,j}$ is the observed value of feature $j$ at time step $t$ in sequence $i$, and $\hat{X}_{i,t,j}$ is the corresponding reconstructed value.

The fixed anomaly threshold is the 99th percentile of these per-sequence errors on the threshold-calibration subset:

$$
\text{THRESHOLD} = P_{99}\left(\left\{\, e(\mathbf{X}_i) : \mathbf{X}_i \in \mathcal{T} \,\right\}\right)
$$

where $\mathcal{T}$ denotes the threshold-calibration subset.

A sequence is flagged as anomalous when its reconstruction error exceeds the threshold:

$$
\text{flag}(\mathbf{X}_i) =
\begin{cases}
1 & \text{if } e(\mathbf{X}_i) > \text{THRESHOLD} \\
0 & \text{otherwise}
\end{cases}
$$

The flag rate on a subset of $N$ sequences is:

$$
\text{flag\_rate} = \frac{1}{N} \sum_{i=1}^{N} \text{flag}(\mathbf{X}_i)
$$

For the threshold-calibration subset, this flag rate is 1% by construction. For the normal-validation subset, it is the direct estimate of false-positive behaviour on later normal data.

For the threshold-calibration subset, this flag rate is 1% by construction. For the normal-validation subset, it is the direct estimate of false-positive behaviour on later normal data.

### Temporal Windowing

The model operates on sequences rather than isolated observations. Each sequence contains a fixed number of consecutive observations while preserving their chronological order.

The primary reference implementation uses a window size of 30 observations. This project will initially retain that value so that the implementation remains comparable to the reference, unless the data or modelling results provide evidence that a change is necessary.

Training sequences are generated from normal calibration data only. Normal-validation sequences are chronologically disjoint from the training sequences. Attack sequences are evaluated separately and are known to overlap the calibration and validation time windows, even though they share no exact timestamps with them. Attack-side evaluation is therefore treated as a distinct, time-window-overlapping evaluation, not as a temporally independent hold-out.

The training, normal-validation and threshold-calibration data use the eligible-normal split established in Notebook 04 §9 as their outer boundary: 316,238 calibration observations (2015-12-28 10:00:00 → 2016-01-01 15:51:19) and 79,060 normal-validation observations (2016-01-01 15:51:20 → 2016-01-02 14:59:59), drawn from the 395,298 eligible normal observations. Within this boundary, the calibration period is further divided chronologically as described in Section 3.2, so that threshold calibration uses data held out from model training. No observation from the normal-validation period contributes to training, model selection, or threshold calibration.

Sequence generation respects the calibration/validation boundary: the last training sequence ends strictly before the first validation observation, so no validation observation contributes to the reconstruction-error statistics on the calibration data. Sequences are generated independently within each chronological subset using the configured stride, so no sequence spans a subset boundary. No validation observation is ever included in a calibration sequence, and no calibration observation is ever included in a validation sequence.

### Input Feature Contract

Notebook 04 §12.5 documented that nine sensors — `P202`, `P204`, `P206`, `P301`, `P401`, `P404`, `P502`, `P601`, `P603` — had zero standard deviation during the calibration period, and Notebook 03 §6.2 identified six of these as globally constant across both the normal and attack datasets. Zero-variance features produce no reconstruction-error signal and contribute no gradient to the autoencoder.

This notebook excludes the nine constant-in-calibration sensors from the LSTM input, giving a 42-feature input contract. This choice mirrors the 42-feature evaluation set used by the simple statistical threshold baseline in Notebook 04 §12.5 and eliminates zero-gradient inputs from the training signal. The 42 retained features are the remaining sensor columns after the nine excluded features are removed. The input contract is recorded here so that the anomaly scores reported subsequently are reproducible.

### Distribution-Shift Risk

Notebook 04 §10.4 found that the rule-based detector, calibrated on the first 80% of the eligible normal data, flagged 92.71% of the remaining 20% of normal observations. This indicates a substantial shift in the marginal distributions of several sensors between the calibration and validation periods. An LSTM autoencoder trained on calibration data may produce systematically higher reconstruction errors on the validation period, in which case a threshold calibrated directly on the training reconstruction errors would over-flag normal validation observations.

To mitigate this risk, the anomaly threshold is calibrated on a portion of the calibration period held out from model training, so that the threshold reflects the model's reconstruction behaviour on data not seen during training. The distribution of reconstruction errors on the normal validation period is reported alongside the threshold so that the extent of any residual shift remains visible.

### Data Leakage Prevention

The preprocessing parameters used by the model must be learned from the normal calibration data only. Validation and attack observations must not influence the fitted scaler or the model parameters during training.

The final anomaly threshold is also established without using attack labels for threshold selection. All threshold values are derived from held-out normal-calibration reconstruction errors.

## 3. Reproducibility and Model Configuration

The LSTM autoencoder is implemented in PyTorch. The initial configuration follows the primary reference notebook to maintain comparability with the reference implementation.

### 3.1 Model Configuration

| Parameter | Value |
|:--|--:|
| Random seed | 42 |
| Sequence window | 30 observations |
| Training stride | 5 observations |
| Evaluation stride | 1 observation |
| Hidden size | 64 |
| LSTM layers | 2 |
| Bottleneck size | 16 |
| Bottleneck activation | LeakyReLU (negative slope 0.01) |
| Decoder projection activation | LeakyReLU (negative slope 0.01) |
| Dropout | 0.2 |
| Loss function | Mean Squared Error |
| Optimizer | Adam |
| Initial learning rate | 0.001 |
| Batch size | 256 |
| Maximum training epochs | 100 |
| Learning-rate scheduler | ReduceLROnPlateau |
| Learning-rate reduction factor | 0.5 |
| Scheduler patience | 3 epochs |
| Early-stopping patience | 10 epochs |

These settings define the initial experiment. They are not assumed to be universally optimal, and any justified changes will be recorded.

### 3.2 Calibration Data Separation

The normal calibration period established in Notebook 04 contains 316,238 observations. This period is divided chronologically into three non-overlapping portions:

1. **Model-training data:** Used to update the model parameters.
2. **Training-monitoring data:** Used to monitor reconstruction loss, support learning-rate scheduling, and determine early stopping.
3. **Threshold-calibration data:** Kept separate from model fitting and training monitoring. Its reconstruction errors determine the fixed 99th-percentile anomaly threshold.

The specific split uses 70 / 15 / 15 proportions of the calibration period:


| Portion | Observations | Share | Time range |
|:--|--:|--:|:--|
| Model training | 221,366 | 70% | 2015-12-28 10:00:00 → 2015-12-31 12:43:42 |
| Training monitoring | 47,435 | 15% | 2015-12-31 12:43:43 → 2016-01-01 02:23:17 |
| Threshold calibration | 47,437 | 15% | 2016-01-01 02:23:18 → 2016-01-01 15:51:19 |

The normal validation period established in Notebook 04 remains separate from these calibration subsets. It is used to assess reconstruction errors and detection behaviour after the model and threshold have been fixed.

Attack-labelled observations are not used for model training, training monitoring, or threshold selection.

### 3.3 Training Procedure and Monitoring

Training reports, for each epoch, the epoch number, training loss, monitoring loss, and current learning rate. The training history — including per-epoch losses and learning rates — is retained for subsequent analysis and visualisation.

The best model state, as determined by the lowest monitoring loss, is preserved. Early stopping halts training when the monitoring loss has not improved for the configured number of patience epochs.

Random seeds are set once in the environment-setup code cell before any model initialisation. Device selection, model configuration, and data-split boundaries are recorded in Sections 2 and 3.1, and are reflected in the training-time printouts so that a completed training run can be reconstructed from its log alone.

Reproducibility verification across devices and software environments is deferred to Notebook 12.

The bottleneck projection and decoder projection use LeakyReLU with negative slope 0.01. This activation was selected after two earlier variants were trained under the same split for comparison: a linear bottleneck (no activation) and a ReLU bottleneck. The ReLU bottleneck produced a monitoring-loss plateau at approximately 0.0069 and did not recover with additional patience or learning-rate reduction — consistent with dying ReLU units in the 16-dimensional bottleneck. The linear bottleneck reached approximately 0.0051. LeakyReLU reached approximately 0.0033. LeakyReLU was retained because it preserves the nonlinear compression the autoencoder design requires without permanently killing bottleneck units. The exact monitoring-loss value from the final run is recorded in Section 15.

### 3.4 Training Environment

Model training is executed on a Google Colab runtime (T4 GPU) accessed from VSCode. The Colab runtime is a remote Linux virtual machine, so the raw CSVs are read from a Google Drive mount (/content/drive/MyDrive/swat-dataset/) rather than from the local laptop filesystem. The runtime is detected in Section 4, and the local data/raw/ path remains a fallback for offline execution.

Artifacts produced by the notebook — the fitted scaler and any sequence arrays or model state required for later evaluation — are written to a dedicated artifacts directory. On Colab this directory is on Google Drive (/content/drive/MyDrive/swat-artifacts/); locally it is ./artifacts/. Sections 16–18 read from this artifacts directory, so the same notebook runs both on Colab and locally without code changes.

The training log records the actual device, PyTorch version, and wall-clock duration. CUDA-based training may produce loss values that differ slightly from CPU training with the same seed, due to non-deterministic parallel reductions in cuDNN. Cross-environment reproducibility is deferred to Notebook 12.

In [1]:
# 4. Environment Setup and Reproducibility

import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


# 1. Reproducibility


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# 2. Detect runtime and set data + artifact roots


# The Colab runtime has /content/ on its filesystem; the local runtime does not.
IS_COLAB = Path("/content").is_dir()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DATA_ROOT = Path("/content/drive/MyDrive/swat-dataset")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/swat-artifacts")
    ENVIRONMENT = "colab"
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (
            directory
            for directory in (CURRENT_DIR, *CURRENT_DIR.parents)
            if (directory / "data" / "raw" / "normal.csv").is_file()
            and (directory / "data" / "raw" / "attack.csv").is_file()
        ),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Could not locate the project root containing "
            "'data/raw/normal.csv' and 'data/raw/attack.csv'. "
            "Open Jupyter from within the project directory "
            "or one of its subdirectories."
        )
    DATA_ROOT = PROJECT_ROOT / "data" / "raw"
    ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
    ENVIRONMENT = "local"

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


# 3. Verify the data is reachable


if not (DATA_ROOT / "normal.csv").is_file():
    raise FileNotFoundError(
        f"Expected 'normal.csv' under {DATA_ROOT}. "
        f"On Colab, upload the raw CSVs to the Drive folder "
        f"'MyDrive/swat-dataset/' first."
    )

if not (DATA_ROOT / "attack.csv").is_file():
    raise FileNotFoundError(
        f"Expected 'attack.csv' under {DATA_ROOT}."
    )


# 4. Select the available PyTorch device


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 5. Report the environment


print("ENVIRONMENT SETUP")
print("-" * 50)
print(f"Runtime:            {ENVIRONMENT}")
print(f"Data root:          {DATA_ROOT}")
print(f"Artifact root:      {ARTIFACT_ROOT}")
print(f"Normal file found:  {(DATA_ROOT / 'normal.csv').is_file()}")
print(f"Attack file found:  {(DATA_ROOT / 'attack.csv').is_file()}")
print()
print(f"Random seed:        {SEED}")
print(f"Python version:     {sys.version.split()[0]}")
print(f"PyTorch version:    {torch.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Pandas version:     {pd.__version__}")
print(f"Selected device:    {DEVICE}")

if DEVICE.type == "cuda":
    print(f"GPU:                {torch.cuda.get_device_name(0)}")
else:
    print("GPU:                No CUDA-compatible GPU available to PyTorch.")
    print("                    Training will use the CPU.")

print("-" * 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ENVIRONMENT SETUP
--------------------------------------------------
Runtime:            colab
Data root:          /content/drive/MyDrive/swat-dataset
Artifact root:      /content/drive/MyDrive/swat-artifacts
Normal file found:  True
Attack file found:  True

Random seed:        42
Python version:     3.12.13
PyTorch version:    2.11.0+cu128
NumPy version:      2.0.2
Pandas version:     2.2.2
Selected device:    cuda
GPU:                Tesla T4
--------------------------------------------------


In [2]:
import warnings

# Suppress the specific date-parsing warning by message text
warnings.filterwarnings("ignore", message=".*Parsing dates.*")

# 1. Load the normal dataset

normal_path = DATA_ROOT / "normal.csv"

normal = pd.read_csv(normal_path, low_memory=False)

# Standardise column names without modifying the raw file.
normal.columns = normal.columns.str.strip()

# Identify the timestamp and label columns.
TIMESTAMP_COL = "Timestamp"

LABEL_COL = next(
    (
        column
        for column in normal.columns
        if column.lower() in {"normal/attack", "label"}
    ),
    None,
)

if TIMESTAMP_COL not in normal.columns:
    raise KeyError("The expected 'Timestamp' column was not found.")

if LABEL_COL is None:
    raise KeyError("The dataset label column was not found.")

# Identify the sensor features.
feature_cols = [
    column
    for column in normal.columns
    if column not in {TIMESTAMP_COL, LABEL_COL}
]

# Parse timestamps and identify complete observations.
normal[TIMESTAMP_COL] = pd.to_datetime(
    normal[TIMESTAMP_COL],
    errors="coerce",
)

complete_sensor_mask = normal[feature_cols].notna().all(axis=1)
valid_timestamp_mask = normal[TIMESTAMP_COL].notna()

eligible_mask = complete_sensor_mask & valid_timestamp_mask

eligible_normal = normal.loc[eligible_mask].copy()

# 2. Reconstruct the exact chronological split from Notebook 04

CALIBRATION_FRACTION = 0.80

n_eligible = len(eligible_normal)
calibration_end = int(n_eligible * CALIBRATION_FRACTION)

normal_calibration = eligible_normal.iloc[:calibration_end].copy()
normal_validation = eligible_normal.iloc[calibration_end:].copy()

# 3. Verify the reconstructed data

print("NORMAL DATA PREPARATION")
print("-" * 60)

print(f"Normal dataset shape:          {normal.shape}")
print(f"Number of sensor features:     {len(feature_cols)}")
print(f"Valid timestamps:              {valid_timestamp_mask.sum():,}")
print(f"Complete sensor observations:  {complete_sensor_mask.sum():,}")
print(f"Eligible observations:         {n_eligible:,}")

print("\nCALIBRATION AND VALIDATION SPLIT")
print("-" * 60)

print(f"Calibration observations:      {len(normal_calibration):,}")
print(f"Validation observations:       {len(normal_validation):,}")

print(
    "Calibration period:           "
    f"{normal_calibration[TIMESTAMP_COL].iloc[0]} "
    "to "
    f"{normal_calibration[TIMESTAMP_COL].iloc[-1]}"
)

print(
    "Validation period:            "
    f"{normal_validation[TIMESTAMP_COL].iloc[0]} "
    "to "
    f"{normal_validation[TIMESTAMP_COL].iloc[-1]}"
)

print("\nVERIFICATION")
print("-" * 60)

print(
    "Calibration precedes validation:",
    normal_calibration[TIMESTAMP_COL].iloc[-1]
    < normal_validation[TIMESTAMP_COL].iloc[0],
)

print(
    "Calibration contains only normal labels:",
    normal_calibration[LABEL_COL].eq("Normal").all(),
)

print(
    "Validation contains only normal labels:",
    normal_validation[LABEL_COL].eq("Normal").all(),
)

print("-" * 60)

NORMAL DATA PREPARATION
------------------------------------------------------------
Normal dataset shape:          (1387098, 53)
Number of sensor features:     51
Valid timestamps:              1,387,093
Complete sensor observations:  395,298
Eligible observations:         395,298

CALIBRATION AND VALIDATION SPLIT
------------------------------------------------------------
Calibration observations:      316,238
Validation observations:       79,060
Calibration period:           2015-12-28 10:00:00 to 2016-01-01 15:51:19
Validation period:            2016-01-01 15:51:20 to 2016-01-02 14:59:59

VERIFICATION
------------------------------------------------------------
Calibration precedes validation: True
Calibration contains only normal labels: True
Validation contains only normal labels: True
------------------------------------------------------------


In [3]:
# 6. Define the LSTM Input Feature Contract

EXCLUDED_FEATURES = [
    "P202",
    "P204",
    "P206",
    "P301",
    "P401",
    "P404",
    "P502",
    "P601",
    "P603",
]


# Verify that every excluded feature exists.
missing_excluded = [
    feature for feature in EXCLUDED_FEATURES
    if feature not in feature_cols
]

if missing_excluded:
    raise KeyError(
        "The following expected zero-variance calibration features "
        f"were not found: {missing_excluded}"
    )

# Retain the remaining sensor features.
model_feature_cols = [
    feature for feature in feature_cols
    if feature not in EXCLUDED_FEATURES
]

# Verify the expected feature count.
if len(model_feature_cols) != 42:
    raise ValueError(
        f"Expected 42 model features, found {len(model_feature_cols)}."
    )


# Confirm that the excluded features have zero variance
# in the normal calibration period.

calibration_std = normal_calibration[EXCLUDED_FEATURES].std(ddof=1)

nonzero_excluded_std = calibration_std[calibration_std > 0]

if not nonzero_excluded_std.empty:
    raise ValueError(
        "One or more excluded features are not constant in the "
        "normal calibration period:\n"
        f"{nonzero_excluded_std}"
    )

print("LSTM INPUT FEATURE CONTRACT")
print("-" * 60)
print(f"Original sensor features:     {len(feature_cols)}")
print(f"Excluded features:            {len(EXCLUDED_FEATURES)}")
print(f"Model input features:         {len(model_feature_cols)}")
print()
print("Excluded features:")
print(", ".join(EXCLUDED_FEATURES))
print()
print("Model input contract verified:", len(model_feature_cols) == 42)

LSTM INPUT FEATURE CONTRACT
------------------------------------------------------------
Original sensor features:     51
Excluded features:            9
Model input features:         42

Excluded features:
P202, P204, P206, P301, P401, P404, P502, P601, P603

Model input contract verified: True


In [4]:
# 7. Internal Calibration Split

TRAIN_FRACTION = 0.70
MONITOR_FRACTION = 0.15
THRESHOLD_FRACTION = 0.15

n_calibration = len(normal_calibration)

train_end = int(n_calibration * TRAIN_FRACTION)
monitor_end = train_end + int(n_calibration * MONITOR_FRACTION)

lstm_train = normal_calibration.iloc[:train_end].copy()
lstm_monitor = normal_calibration.iloc[train_end:monitor_end].copy()
lstm_threshold = normal_calibration.iloc[monitor_end:].copy()

print("LSTM INTERNAL CALIBRATION SPLIT")
print("-" * 60)

print(f"Total calibration observations: {n_calibration:,}")
print()

print(
    f"Training observations:          {len(lstm_train):,} "
    f"({len(lstm_train) / n_calibration:.2%})"
)
print(
    f"Monitoring observations:        {len(lstm_monitor):,} "
    f"({len(lstm_monitor) / n_calibration:.2%})"
)
print(
    f"Threshold-calibration obs.:     {len(lstm_threshold):,} "
    f"({len(lstm_threshold) / n_calibration:.2%})"
)

print()
print("TIME PERIODS")
print("-" * 60)

print(
    "Training:          "
    f"{lstm_train[TIMESTAMP_COL].iloc[0]} "
    f"to {lstm_train[TIMESTAMP_COL].iloc[-1]}"
)

print(
    "Monitoring:        "
    f"{lstm_monitor[TIMESTAMP_COL].iloc[0]} "
    f"to {lstm_monitor[TIMESTAMP_COL].iloc[-1]}"
)

print(
    "Threshold:         "
    f"{lstm_threshold[TIMESTAMP_COL].iloc[0]} "
    f"to {lstm_threshold[TIMESTAMP_COL].iloc[-1]}"
)

print()
print("BOUNDARY VERIFICATION")
print("-" * 60)

print(
    "Training precedes monitoring:",
    lstm_train[TIMESTAMP_COL].iloc[-1]
    < lstm_monitor[TIMESTAMP_COL].iloc[0]
)

print(
    "Monitoring precedes threshold calibration:",
    lstm_monitor[TIMESTAMP_COL].iloc[-1]
    < lstm_threshold[TIMESTAMP_COL].iloc[0]
)

print(
    "All three sets contain only normal labels:",
    (
        lstm_train[LABEL_COL].eq("Normal").all()
        and lstm_monitor[LABEL_COL].eq("Normal").all()
        and lstm_threshold[LABEL_COL].eq("Normal").all()
    )
)

LSTM INTERNAL CALIBRATION SPLIT
------------------------------------------------------------
Total calibration observations: 316,238

Training observations:          221,366 (70.00%)
Monitoring observations:        47,435 (15.00%)
Threshold-calibration obs.:     47,437 (15.00%)

TIME PERIODS
------------------------------------------------------------
Training:          2015-12-28 10:00:00 to 2015-12-31 12:43:42
Monitoring:        2015-12-31 12:43:43 to 2016-01-01 02:23:17
Threshold:         2016-01-01 02:23:18 to 2016-01-01 15:51:19

BOUNDARY VERIFICATION
------------------------------------------------------------
Training precedes monitoring: True
Monitoring precedes threshold calibration: True
All three sets contain only normal labels: True


In [5]:
# 8. Fit the Scaler on Training Data Only

from sklearn.preprocessing import MinMaxScaler

# Extract the 42 model features from each chronological subset.
X_train_raw = lstm_train[model_feature_cols].astype(np.float32)
X_monitor_raw = lstm_monitor[model_feature_cols].astype(np.float32)
X_threshold_raw = lstm_threshold[model_feature_cols].astype(np.float32)
X_validation_raw = normal_validation[model_feature_cols].astype(np.float32)

# Confirm that the selected inputs contain no missing or infinite values.

datasets_to_check = {
    "Training": X_train_raw,
    "Monitoring": X_monitor_raw,
    "Threshold calibration": X_threshold_raw,
    "Normal validation": X_validation_raw,
}

for dataset_name, dataset in datasets_to_check.items():
    values = dataset.to_numpy()

    if not np.isfinite(values).all():
        raise ValueError(
            f"{dataset_name} features contain missing or infinite values."
        )

# Fit only on the model-training subset.
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_raw)

# Apply the fitted scaler without refitting.
X_monitor = scaler.transform(X_monitor_raw)
X_threshold = scaler.transform(X_threshold_raw)
X_validation = scaler.transform(X_validation_raw)

# Verify shapes and scaling behaviour.
print("SCALING VERIFICATION")
print("-" * 60)

print(f"Training array shape:            {X_train.shape}")
print(f"Monitoring array shape:          {X_monitor.shape}")
print(f"Threshold-calibration shape:     {X_threshold.shape}")
print(f"Normal-validation shape:         {X_validation.shape}")

print()
print(
    "Training values within [0, 1]:",
    bool((X_train >= 0).all() and (X_train <= 1).all())
)

print(
    "Monitoring values outside [0, 1]:",
    int(((X_monitor < 0) | (X_monitor > 1)).sum())
)

print(
    "Threshold values outside [0, 1]:",
    int(((X_threshold < 0) | (X_threshold > 1)).sum())
)

print(
    "Normal-validation values outside [0, 1]:",
    int(((X_validation < 0) | (X_validation > 1)).sum())
)

print()
print("Scaler fitted on training data only: True")
print("Feature count consistent across subsets:", 
      X_train.shape[1] == X_monitor.shape[1]
      == X_threshold.shape[1] == X_validation.shape[1])


# Save the fitted scaler to the artifacts directory.
import joblib

SCALER_PATH = ARTIFACT_ROOT / "scaler.joblib"
joblib.dump(scaler, SCALER_PATH)
print(f"Scaler saved to: {SCALER_PATH}")

SCALING VERIFICATION
------------------------------------------------------------
Training array shape:            (221366, 42)
Monitoring array shape:          (47435, 42)
Threshold-calibration shape:     (47437, 42)
Normal-validation shape:         (79060, 42)

Training values within [0, 1]: True
Monitoring values outside [0, 1]: 71582
Threshold values outside [0, 1]: 114654
Normal-validation values outside [0, 1]: 261512

Scaler fitted on training data only: True
Feature count consistent across subsets: True
Scaler saved to: /content/drive/MyDrive/swat-artifacts/scaler.joblib


In [6]:
# 9. Define Sequence-Generation Parameters

WINDOW_SIZE = 30
TRAIN_STRIDE = 5
EVAL_STRIDE = 1

# Basic parameter checks.
if WINDOW_SIZE <= 0:
    raise ValueError("WINDOW_SIZE must be positive.")

if TRAIN_STRIDE <= 0 or EVAL_STRIDE <= 0:
    raise ValueError("Sequence strides must be positive.")

if X_train.shape[1] != len(model_feature_cols):
    raise ValueError("Training feature count does not match the input contract.")

if any(
    len(array) < WINDOW_SIZE
    for array in (X_train, X_monitor, X_threshold, X_validation)
):
    raise ValueError("At least one subset is shorter than WINDOW_SIZE.")

print("SEQUENCE-GENERATION CONFIGURATION")
print("-" * 60)
print(f"Window size:              {WINDOW_SIZE}")
print(f"Training stride:           {TRAIN_STRIDE}")
print(f"Evaluation stride:         {EVAL_STRIDE}")
print(f"Features per observation:  {len(model_feature_cols)}")
print(f"Expected sequence shape:   (number of sequences, {WINDOW_SIZE}, {len(model_feature_cols)})")
print()
print("All subsets are long enough for sequence generation.")
print("Sequence parameters verified.")

SEQUENCE-GENERATION CONFIGURATION
------------------------------------------------------------
Window size:              30
Training stride:           5
Evaluation stride:         1
Features per observation:  42
Expected sequence shape:   (number of sequences, 30, 42)

All subsets are long enough for sequence generation.
Sequence parameters verified.


In [7]:
# 10. Sequence-Generation Function

def create_sequences(data, window_size, stride):
    """
    Create chronological sliding-window sequences.

    Parameters
    ----------
    data : np.ndarray
        Scaled observations with shape (n_observations, n_features).
    window_size : int
        Number of observations in each sequence.
    stride : int
        Step between consecutive sequence starting positions.

    Returns
    -------
    sequences : np.ndarray
        Array with shape (n_sequences, window_size, n_features).
    end_indices : np.ndarray
        Original subset indices corresponding to each sequence's
        final observation.
    """
    data = np.asarray(data)

    if data.ndim != 2:
        raise ValueError(
            f"Expected 2D input, received array with shape {data.shape}."
        )

    if window_size <= 0:
        raise ValueError("window_size must be positive.")

    if stride <= 0:
        raise ValueError("stride must be positive.")

    n_observations, n_features = data.shape

    if n_observations < window_size:
        raise ValueError(
            f"Input contains {n_observations} observations, "
            f"but window_size is {window_size}."
        )

    start_indices = np.arange(
        0,
        n_observations - window_size + 1,
        stride,
    )

    sequences = np.stack(
        [
            data[start:start + window_size]
            for start in start_indices
        ],
        axis=0,
    )

    end_indices = start_indices + window_size - 1

    return sequences.astype(np.float32), end_indices

In [8]:
# 11. Create LSTM Sequences

X_train_seq, train_end_indices = create_sequences(
    X_train,
    window_size=WINDOW_SIZE,
    stride=TRAIN_STRIDE,
)

X_monitor_seq, monitor_end_indices = create_sequences(
    X_monitor,
    window_size=WINDOW_SIZE,
    stride=EVAL_STRIDE,
)

X_threshold_seq, threshold_end_indices = create_sequences(
    X_threshold,
    window_size=WINDOW_SIZE,
    stride=EVAL_STRIDE,
)

X_validation_seq, validation_end_indices = create_sequences(
    X_validation,
    window_size=WINDOW_SIZE,
    stride=EVAL_STRIDE,
)

print("LSTM SEQUENCE CONSTRUCTION")
print("-" * 60)

print(f"Training sequences:             {len(X_train_seq):,}")
print(f"Monitoring sequences:           {len(X_monitor_seq):,}")
print(f"Threshold-calibration sequences:{len(X_threshold_seq):,}")
print(f"Normal-validation sequences:    {len(X_validation_seq):,}")

print()
print("SEQUENCE SHAPES")
print("-" * 60)

print(f"Training:          {X_train_seq.shape}")
print(f"Monitoring:        {X_monitor_seq.shape}")
print(f"Threshold:         {X_threshold_seq.shape}")
print(f"Normal validation: {X_validation_seq.shape}")

print()
print("SEQUENCE INTEGRITY")
print("-" * 60)

print(
    "Training shape correct:",
    X_train_seq.shape[1:] == (WINDOW_SIZE, len(model_feature_cols))
)

print(
    "Monitoring shape correct:",
    X_monitor_seq.shape[1:] == (WINDOW_SIZE, len(model_feature_cols))
)

print(
    "Threshold shape correct:",
    X_threshold_seq.shape[1:] == (WINDOW_SIZE, len(model_feature_cols))
)

print(
    "Validation shape correct:",
    X_validation_seq.shape[1:] == (WINDOW_SIZE, len(model_feature_cols))
)

print()
print("First training sequence ends at subset index:", train_end_indices[0])
print("Last training sequence ends at subset index:", train_end_indices[-1])

print(
    "First monitoring sequence ends at subset index:",
    monitor_end_indices[0]
)
print(
    "Last monitoring sequence ends at subset index:",
    monitor_end_indices[-1]
)

print(
    "First threshold sequence ends at subset index:",
    threshold_end_indices[0]
)
print(
    "Last threshold sequence ends at subset index:",
    threshold_end_indices[-1]
)

print(
    "First validation sequence ends at subset index:",
    validation_end_indices[0]
)
print(
    "Last validation sequence ends at subset index:",
    validation_end_indices[-1]
)

LSTM SEQUENCE CONSTRUCTION
------------------------------------------------------------
Training sequences:             44,268
Monitoring sequences:           47,406
Threshold-calibration sequences:47,408
Normal-validation sequences:    79,031

SEQUENCE SHAPES
------------------------------------------------------------
Training:          (44268, 30, 42)
Monitoring:        (47406, 30, 42)
Threshold:         (47408, 30, 42)
Normal validation: (79031, 30, 42)

SEQUENCE INTEGRITY
------------------------------------------------------------
Training shape correct: True
Monitoring shape correct: True
Threshold shape correct: True
Validation shape correct: True

First training sequence ends at subset index: 29
Last training sequence ends at subset index: 221364
First monitoring sequence ends at subset index: 29
Last monitoring sequence ends at subset index: 47434
First threshold sequence ends at subset index: 29
Last threshold sequence ends at subset index: 47436
First validation sequence en

In [9]:
# 11b. Save evaluation sequences to the artifacts directory.

SEQUENCES_PATH = ARTIFACT_ROOT / "evaluation_sequences.npz"

np.savez_compressed(
    SEQUENCES_PATH,
    X_threshold_seq=X_threshold_seq,
    X_validation_seq=X_validation_seq,
    threshold_end_indices=threshold_end_indices,
    validation_end_indices=validation_end_indices,
)

print(f"Evaluation sequences saved to: {SEQUENCES_PATH}")

Evaluation sequences saved to: /content/drive/MyDrive/swat-artifacts/evaluation_sequences.npz


In [11]:
# 12. LSTM Autoencoder Architecture
import torch.nn as nn
import torch.nn.functional as F

INPUT_SIZE = len(model_feature_cols)
HIDDEN_SIZE = 64
NUM_LAYERS = 2
BOTTLENECK_SIZE = 16
DROPOUT = 0.2


class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        bottleneck_size,
        dropout,
    ):
        super().__init__()

        self.encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.encoder_projection = nn.Linear(
            hidden_size,
            bottleneck_size,
        )

        self.decoder_projection = nn.Linear(
            bottleneck_size,
            hidden_size,
        )

        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.output_layer = nn.Linear(
            hidden_size,
            input_size,
        )

    def forward(self, x):
        # Encode the input sequence.
        _, (hidden, _) = self.encoder(x)

        # Compress the final hidden state into the bottleneck.
        latent = F.leaky_relu(self.encoder_projection(hidden[-1]), negative_slope=0.01)

        # Expand the bottleneck across the sequence length.
        decoder_input = F.leaky_relu(self.decoder_projection(latent), negative_slope=0.01)
        decoder_input = decoder_input.unsqueeze(1).repeat(
            1,
            x.size(1),
            1,
        )

        # Decode and reconstruct the input features.
        decoded, _ = self.decoder(decoder_input)
        reconstruction = self.output_layer(decoded)

        return reconstruction


model = LSTMAutoencoder(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    bottleneck_size=BOTTLENECK_SIZE,
    dropout=DROPOUT,
).to(DEVICE)

print("LSTM AUTOENCODER")
print("-" * 60)
print(f"Input features:       {INPUT_SIZE}")
print(f"Hidden size:          {HIDDEN_SIZE}")
print(f"LSTM layers:          {NUM_LAYERS}")
print(f"Bottleneck size:      {BOTTLENECK_SIZE}")
print(f"Dropout:              {DROPOUT}")
print(f"Device:               {DEVICE}")
print()
print(model)

LSTM AUTOENCODER
------------------------------------------------------------
Input features:       42
Hidden size:          64
LSTM layers:          2
Bottleneck size:      16
Dropout:              0.2
Device:               cuda

LSTMAutoencoder(
  (encoder): LSTM(42, 64, num_layers=2, batch_first=True, dropout=0.2)
  (encoder_projection): Linear(in_features=64, out_features=16, bias=True)
  (decoder_projection): Linear(in_features=16, out_features=64, bias=True)
  (decoder): LSTM(64, 64, num_layers=2, batch_first=True, dropout=0.2)
  (output_layer): Linear(in_features=64, out_features=42, bias=True)
)


In [12]:
# 13. Training Configuration and DataLoaders

from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 256
MAX_EPOCHS = 100
LEARNING_RATE = 1e-3
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3
EARLY_STOPPING_PATIENCE = 10

# Convert sequence arrays to PyTorch tensors.
X_train_tensor = torch.from_numpy(X_train_seq)
X_monitor_tensor = torch.from_numpy(X_monitor_seq)

# Create datasets. Autoencoders use the input as the reconstruction target.
train_dataset = TensorDataset(X_train_tensor)
monitor_dataset = TensorDataset(X_monitor_tensor)

# Training batches are shuffled; chronology within each sequence is preserved.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
)

# Monitoring order is retained.
monitor_loader = DataLoader(
    monitor_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
)

# Define loss and optimizer.
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
)

print("TRAINING CONFIGURATION")
print("-" * 60)
print(f"Batch size:                  {BATCH_SIZE}")
print(f"Maximum epochs:              {MAX_EPOCHS}")
print(f"Initial learning rate:       {LEARNING_RATE}")
print(f"Loss function:               {criterion}")
print(f"Optimizer:                   Adam")
print(f"Scheduler factor:            {SCHEDULER_FACTOR}")
print(f"Scheduler patience:          {SCHEDULER_PATIENCE}")
print(f"Early-stopping patience:     {EARLY_STOPPING_PATIENCE}")
print()
print("TRAINING DATA")
print("-" * 60)
print(f"Training sequences:          {len(train_dataset):,}")
print(f"Monitoring sequences:        {len(monitor_dataset):,}")
print(f"Training batches per epoch:  {len(train_loader):,}")
print(f"Monitoring batches per epoch:{len(monitor_loader):,}")
print()
print("Threshold-calibration sequences remain separate:", 
      X_threshold_seq.shape[0] > 0)

TRAINING CONFIGURATION
------------------------------------------------------------
Batch size:                  256
Maximum epochs:              100
Initial learning rate:       0.001
Loss function:               MSELoss()
Optimizer:                   Adam
Scheduler factor:            0.5
Scheduler patience:          3
Early-stopping patience:     10

TRAINING DATA
------------------------------------------------------------
Training sequences:          44,268
Monitoring sequences:        47,406
Training batches per epoch:  173
Monitoring batches per epoch:186

Threshold-calibration sequences remain separate: True


In [13]:
# 14. Train the LSTM Autoencoder

import copy
import time

training_loss_history = []
monitoring_loss_history = []
learning_rate_history = []

best_monitoring_loss = float("inf")
best_epoch = 0
best_model_state = None
epochs_without_improvement = 0

training_start_time = time.time()

print("LSTM AUTOENCODER TRAINING")
print("=" * 75)

for epoch in range(1, MAX_EPOCHS + 1):
    # -----------------------------
    # Training phase
    # -----------------------------
    model.train()
    running_training_loss = 0.0
    training_sequence_count = 0

    for (batch,) in train_loader:
        batch = batch.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        reconstruction = model(batch)
        loss = criterion(reconstruction, batch)

        loss.backward()
        optimizer.step()

        batch_size_actual = batch.size(0)
        running_training_loss += loss.item() * batch_size_actual
        training_sequence_count += batch_size_actual

    epoch_training_loss = (
        running_training_loss / training_sequence_count
    )

    # -----------------------------
    # Monitoring phase
    # -----------------------------
    model.eval()
    running_monitoring_loss = 0.0
    monitoring_sequence_count = 0

    with torch.no_grad():
        for (batch,) in monitor_loader:
            batch = batch.to(DEVICE)

            reconstruction = model(batch)
            loss = criterion(reconstruction, batch)

            batch_size_actual = batch.size(0)
            running_monitoring_loss += loss.item() * batch_size_actual
            monitoring_sequence_count += batch_size_actual

    epoch_monitoring_loss = (
        running_monitoring_loss / monitoring_sequence_count
    )

    # Record metrics and learning rate for this epoch.
    current_learning_rate = optimizer.param_groups[0]["lr"]

    training_loss_history.append(epoch_training_loss)
    monitoring_loss_history.append(epoch_monitoring_loss)
    learning_rate_history.append(current_learning_rate)

    # Update the scheduler using monitoring loss.
    scheduler.step(epoch_monitoring_loss)

    # Keep the model state with the lowest monitoring loss.
    if epoch_monitoring_loss < best_monitoring_loss:
        best_monitoring_loss = epoch_monitoring_loss
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # Report progress.
    elapsed_minutes = (time.time() - training_start_time) / 60

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train loss: {epoch_training_loss:.6f} | "
        f"Monitoring loss: {epoch_monitoring_loss:.6f} | "
        f"LR: {current_learning_rate:.2e} | "
        f"Best epoch: {best_epoch:02d} | "
        f"Elapsed: {elapsed_minutes:.1f} min"
    )

    # Stop if monitoring loss has not improved for the configured patience.
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"\nEarly stopping triggered at epoch {epoch}. "
            f"No monitoring-loss improvement for "
            f"{EARLY_STOPPING_PATIENCE} consecutive epochs."
        )
        break

# Restore the best model state.
if best_model_state is None:
    raise RuntimeError(
        "Training did not produce a valid best model state."
    )

model.load_state_dict(best_model_state)
model.eval()

training_duration_minutes = (
    time.time() - training_start_time
) / 60

print("\n" + "=" * 75)
print("TRAINING COMPLETE")
print("-" * 75)
print(f"Epochs completed:          {len(training_loss_history)}")
print(f"Best epoch:                {best_epoch}")
print(f"Best monitoring loss:      {best_monitoring_loss:.8f}")
print(f"Final recorded train loss: {training_loss_history[-1]:.8f}")
print(f"Final recorded monitor loss: {monitoring_loss_history[-1]:.8f}")
print(f"Training duration:         {training_duration_minutes:.2f} minutes")
print("Best model state restored: True")


# Save the trained model state to the artifacts directory.
MODEL_PATH = ARTIFACT_ROOT / "lstm_autoencoder.pth"
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model state saved to: {MODEL_PATH}")

LSTM AUTOENCODER TRAINING
Epoch 01/100 | Train loss: 0.086508 | Monitoring loss: 0.055172 | LR: 1.00e-03 | Best epoch: 01 | Elapsed: 0.0 min
Epoch 02/100 | Train loss: 0.038218 | Monitoring loss: 0.039809 | LR: 1.00e-03 | Best epoch: 02 | Elapsed: 0.1 min
Epoch 03/100 | Train loss: 0.020271 | Monitoring loss: 0.032361 | LR: 1.00e-03 | Best epoch: 03 | Elapsed: 0.1 min
Epoch 04/100 | Train loss: 0.017641 | Monitoring loss: 0.027335 | LR: 1.00e-03 | Best epoch: 04 | Elapsed: 0.1 min
Epoch 05/100 | Train loss: 0.011662 | Monitoring loss: 0.024473 | LR: 1.00e-03 | Best epoch: 05 | Elapsed: 0.2 min
Epoch 06/100 | Train loss: 0.008695 | Monitoring loss: 0.017773 | LR: 1.00e-03 | Best epoch: 06 | Elapsed: 0.2 min
Epoch 07/100 | Train loss: 0.007109 | Monitoring loss: 0.014382 | LR: 1.00e-03 | Best epoch: 07 | Elapsed: 0.2 min
Epoch 08/100 | Train loss: 0.006221 | Monitoring loss: 0.011863 | LR: 1.00e-03 | Best epoch: 08 | Elapsed: 0.2 min
Epoch 09/100 | Train loss: 0.005486 | Monitoring loss:

## 15. Training Findings

The LSTM autoencoder trained for all 100 configured epochs without triggering early stopping. The lowest monitoring reconstruction loss was `0.00334621`, reached at epoch 100 — the final epoch — and the corresponding model state was restored after training and saved to `lstm_autoencoder.pth` in the artifacts directory.

Training reconstruction loss decreased from `0.086508` at epoch 1 to `0.00090017` at epoch 100. Monitoring reconstruction loss decreased from `0.055172` at epoch 1 to a minimum of `0.00334621` at epoch 100. The monitoring loss showed local fluctuations throughout training but the overall trend was downward across the full run.

The learning-rate scheduler reduced the learning rate five times: from `0.001` to `0.0005` at epoch 56, to `0.00025` at epoch 63, to `0.000125` at epoch 72, to `0.0000625` at epoch 84, and to `0.00003125` at epoch 98. Each reduction was followed by a period of genuine improvement in monitoring loss, consistent with the intended behaviour of the `ReduceLROnPlateau` schedule. Early stopping was not triggered because the model continued to find marginal improvements within every 10-epoch patience window; the improvement at epoch 100 reset the patience counter and allowed training to complete all 100 configured epochs.

The last twenty-one epochs produced diminishing returns: monitoring loss improved from `0.003360` at epoch 79 to `0.003346` at epoch 100 — a total reduction of approximately 0.4% over 21 epochs. The model is effectively at the noise floor of reconstruction error under the current architecture and training configuration. Extending training further would yield negligible improvement.

The final training loss (`0.00090017`) was approximately 3.7× lower than the final monitoring loss (`0.00334621`). This train-monitor gap is recorded as an observed training characteristic. It is expected under the chronological split, in which the monitoring subset contains later observations not seen during training and subject to the distribution shift documented in Section 2 (Notebook 04 §10.4). The gap is not interpreted as evidence of failure or success in anomaly detection.

The model was trained with LeakyReLU activations in the bottleneck and decoder projections, with a maximum of 100 epochs and an early-stopping patience of 10. The activation choice and its rationale are documented in Sections 3.1 and 3.3.

At this stage, these results demonstrate that the autoencoder learned to reconstruct the normal training sequences and achieved materially lower reconstruction error on the held-out monitoring subset than at the start of training. They do not by themselves establish anomaly-detection performance.

The next step is to calculate reconstruction errors on the separate threshold-calibration, normal-validation, and later attack sequences. The threshold will be derived from the held-out normal threshold-calibration errors at the 99th percentile and will not be optimized using attack labels.

In [ ]:
def calculate_reconstruction_errors(model, sequences, device, batch_size=512):
    """
    Calculate one mean squared reconstruction error per sequence.

    Parameters
    ----------
    model : torch.nn.Module
        Trained LSTM Autoencoder.
    sequences : numpy.ndarray
        Sequences with shape (n_sequences, window_size, n_features).
    device : torch.device
        Device used for model inference.
    batch_size : int
        Number of sequences processed per batch.

    Returns
    -------
    np.ndarray
        One reconstruction error per input sequence.
    """
    model.eval()
    scores = []

    with torch.no_grad():
        for start in range(0, len(sequences), batch_size):
            batch = torch.as_tensor(
                np.asarray(
                    sequences[start:start + batch_size],
                    dtype=np.float32
                ),
                dtype=torch.float32,
                device=device
            )

            reconstruction = model(batch)

            # One MSE score per sequence, averaged across
            # all time steps and sensor features.
            batch_scores = (
                (reconstruction - batch) ** 2
            ).mean(dim=(1, 2))
            